## Historical Feature Ablations

Throughout the classification experiments I used features derived from four preceding recorded earnings events. I wanted to determine how much of the predictive information actually came from the older events, especially considering how well the Previous Outcome baseline performed.

I designed an ablation experiment in which I restricted the historical information available to XGBoost. I also wanted to examine whether the engineered change and trend features contributed anything beyond the historical values they were calculated from.

I kept the original XGBoost hyperparameters fixed throughout these experiments and continued to exclude the reserved 2024 holdout.

In [1]:
import json
import pandas as pd

from IPython.display import display
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results" / "ablations"

ablation_path = RESULTS_DIR / "xgboost_feature_ablations.json"

with ablation_path.open(encoding="utf-8") as file:
    ablation_results = json.load(file)


print(
    "Validation years:", [fold["validation_year"] for fold in ablation_results["folds"]]
)

print("\nFeature configurations:")

for name, columns in ablation_results["feature_sets"].items():
    print(f"{name}: {len(columns)} original features")

print(f"\nReserved holdout samples: {ablation_results['holdout_samples']:,}")
print(f"Holdout used: {ablation_results['holdout_used']}")

Validation years: [2020, 2021, 2022, 2023]

Feature configurations:
full_history: 25 original features
latest_event_only: 6 original features
without_trends: 22 original features

Reserved holdout samples: 7,996
Holdout used: False


### Feature Configurations

I used the Full History configuration as my reference point. It includes all 25 original input features derived from the four preceding recorded earnings events.

For Latest Event Only I retained six features describing the most recent preceding event. These included its actual and estimated EPS, earnings surprise and beat indicator. I also kept the report month and quarter. Features describing older events and any summaries calculated across multiple events were excluded.

The Without Trends configuration retained the four-event history but excluded the following features:

- `eps_change1_2`
- `eps_trend4`
- `surprise_trend4`

I wanted to see whether XGBoost could make effective use of the underlying historical values without these additional calculations. Removing them left 22 original input features.

All three configurations used the same eligible observations. This was important because allowing the Latest Event Only configuration to use observations with shorter earnings histories would have changed the population being evaluated.

In [2]:
feature_sets = ablation_results["feature_sets"]

full_history_features = feature_sets["full_history"]
latest_event_features = feature_sets["latest_event_only"]
without_trends_features = feature_sets["without_trends"]

removed_trend_features = [
    feature
    for feature in full_history_features
    if feature not in without_trends_features
]

print("Latest-event-only features:")

for feature in latest_event_features:
    print(f"  {feature}")

print("\nFeatures removed in the without-trends configuration:")

for feature in removed_trend_features:
    print(f"  {feature}")

Latest-event-only features:
  actual_eps_lag1
  estimate_eps_lag1
  surprise_eps_lag1
  beat_lag1
  previous_report_month
  previous_report_quarter

Features removed in the without-trends configuration:
  eps_change1_2
  eps_trend4
  surprise_trend4


### Performance Comparison

I trained XGBoost separately for each feature configuration using the chronological validation folds from 2020 through 2023. The preprocessing pipeline was fitted on the training data within each fold.

I did not apply mutual-information selection or change the model's hyperparameters during these experiments.

I compared the mean macro F1 across the four validation years and examined accuracy and ROC-AUC to see whether any improvements extended beyond my primary evaluation metric.

In [3]:
configuration_names = {
    "full_history": "Full History",
    "latest_event_only": "Latest Event Only",
    "without_trends": "Without Trends",
}

performance_rows = []

for name, label in configuration_names.items():
    metrics = ablation_results["mean_performance"][name]

    performance_rows.append(
        {
            "Configuration": label,
            "Macro F1": metrics["macro_f1"],
            "Accuracy": metrics["accuracy"],
            "ROC-AUC": metrics["roc_auc"],
        }
    )

performance_table = pd.DataFrame(performance_rows)

display(performance_table.round(4))

,Configuration,Macro F1,Accuracy,ROC-AUC
0,Full History,0.6252,0.6689,0.6946
1,Latest Event Only,0.6133,0.6604,0.6759
2,Without Trends,0.6254,0.6689,0.6938


#### Interpretation

Using the full four-event history increased mean macro F1 from 0.6133 to 0.6252 compared with using only the most recent event. Accuracy also increased from 66.04% to 66.89%, while ROC-AUC improved from 0.6759 to 0.6946.

This was an encouraging result because it suggests that the additional historical information contributed to XGBoost's predictions. The improvement was relatively modest but was apparent across each of the evaluation metrics.

I found considerably less difference when removing the engineered change and trend features. Macro F1 increased by approximately 0.0002 and ROC-AUC decreased slightly. Accuracy remained almost unchanged.

Since the Without Trends configuration still contained the individual historical EPS and surprise values, XGBoost may have been able to learn some of the relationships represented by the removed features directly from those inputs.

### Performance Across Validation Years

The average scores suggested that including older earnings events was beneficial, but I wanted to check whether this improvement occurred consistently or was driven by performance during a particular year.

I examined macro F1 for each feature configuration separately across the four chronological validation folds.

In [4]:
yearly_rows = []

for fold in ablation_results["folds"]:
    models = fold["models"]

    yearly_rows.append(
        {
            "Year": fold["validation_year"],
            "Full History": models["full_history"]["macro_f1"],
            "Latest Event Only": models["latest_event_only"]["macro_f1"],
            "Without Trends": models["without_trends"]["macro_f1"],
        }
    )

yearly_table = pd.DataFrame(yearly_rows)

display(yearly_table.round(4))

,Year,Full History,Latest Event Only,Without Trends
0,2020,0.6070,0.5976,0.6079
1,2021,0.6582,0.6443,0.6573
2,2022,0.6210,0.6126,0.6216
3,2023,0.6146,0.5988,0.6148


#### Interpretation

Full History performed better than Latest Event Only in every validation year. The largest improvement occurred in 2023 when macro F1 increased from 0.5988 to 0.6146. Even in 2022, when the difference was smallest, including the older earnings events improved macro F1 by approximately 0.0084.

I found this consistency more interesting than the average improvement alone. Although classification performance varied between years, the additional historical information remained useful under the conditions I tested.

Removing the trend features produced very little change in any of the validation folds. Without Trends achieved a slightly higher macro F1 in 2020, 2022 and 2023, while Full History performed marginally better in 2021. All of these differences were smaller than 0.001.

I found little evidence that the explicit change and trend calculations were necessary for XGBoost in this experiment.

### Conclusion

One of the main questions I wanted to answer through this project was whether examining a longer earnings history could help predict a company's next earnings outcome.

The ablation results provide some support for this idea. XGBoost achieved higher macro F1 in every validation year when given features derived from four preceding recorded earnings events instead of just the most recent event. The average improvement was approximately 0.0119.

I cannot determine from this experiment which individual historical features were responsible for the improvement because several lagged and summary features were removed together.

I also found that I could remove the three explicitly engineered change and trend features without substantially affecting predictive performance. Given these results I selected the Without Trends configuration for my final evaluation.

This configuration retains information from all four preceding recorded earnings events using 22 original input features. I will use the original XGBoost hyperparameters and evaluate the selected model on the reserved 2024 observations without making further adjustments based on the holdout results.